# 02 — Explore Chirp 3 ASR quality on ROG

Loads a `predictions.jsonl` from `data/7_asr_runs/<RUN_ID>/`, computes
WER/CER (raw and normalised) per instance, and renders three triaging
sections — best N / worst N / random N — each row stacked as
audio ▸ gold ▸ ASR ▸ metrics for an easy ear-plus-eye scan.

Green-light rule: WER_norm on the 170-subset ≤ ~0.25 → proceed to full ROG.

In [ ]:
import json
import random
import re
import string
from pathlib import Path

import jiwer
from IPython.display import Audio, Markdown, display

In [ ]:
HERE = Path.cwd()
if HERE.name != "7_open_router_asr":
    candidate = HERE / "scripts/7_open_router_asr"
    if candidate.exists():
        HERE = candidate
ROOT = HERE.parent.parent

RUN_ID = "chirp3_subset170_2026-11-23_130456"
PRED_PATH = ROOT / "data" / "7_asr_runs" / RUN_ID / "predictions.jsonl"

N_SHOW = 10      # per-bucket count (best / worst / random)
RANDOM_SEED = 42

## Load predictions

In [ ]:
def load_predictions(path: Path) -> list[dict]:
    out = []
    if not path.exists():
        print(f"  (not found: {path})")
        return out
    with open(path, encoding="utf-8") as f:
        for line in f:
            out.append(json.loads(line))
    return out


preds = load_predictions(PRED_PATH)
print(f"{len(preds)} predictions loaded from {PRED_PATH.relative_to(ROOT)}")
n_err = sum(1 for p in preds if p.get("error"))
n_empty = sum(1 for p in preds if not p.get("transcript"))
print(f"  errors: {n_err}   empty transcripts: {n_empty}")

## Text normalisation + per-row WER/CER

Raw metrics compare strings as returned. Normalised metrics lowercase,
strip punctuation (including Slovenian quotes and the ellipsis char),
collapse whitespace. The normalised number is the one to judge against
the 0.25 green-light threshold; raw stays close to it for clean inputs
but inflates on comma/period/case-only differences.

In [ ]:
_PUNCT = string.punctuation + "„“”‚‘’«»…–—"
_PUNCT_RE = re.compile(f"[{re.escape(_PUNCT)}]")

def normalise(s: str) -> str:
    if not s:
        return ""
    s = s.lower()
    s = _PUNCT_RE.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def safe_wer_cer(gold: str, asr: str):
    gold = gold or ""
    asr = asr or ""
    if not gold.strip() and not asr.strip():
        return 0.0, 0.0
    if not gold.strip():
        return float("nan"), float("nan")
    try:
        return jiwer.wer(gold, asr), jiwer.cer(gold, asr)
    except Exception:
        return float("nan"), float("nan")


for p in preds:
    g, a = p.get("text_gold") or "", p.get("transcript") or ""
    p["gold_norm"] = normalise(g)
    p["asr_norm"]  = normalise(a)
    p["wer_raw"], p["cer_raw"]   = safe_wer_cer(g, a)
    p["wer_norm"], p["cer_norm"] = safe_wer_cer(p["gold_norm"], p["asr_norm"])

## Aggregate numbers

In [ ]:
scored = [p for p in preds if not p.get("error") and (p.get("text_gold") or "").strip()]
golds       = [p["text_gold"] for p in scored]
asrs        = [p["transcript"] or "" for p in scored]
golds_norm  = [p["gold_norm"] for p in scored]
asrs_norm   = [p["asr_norm"]  for p in scored]

wer_raw_micro  = jiwer.wer(golds, asrs)
cer_raw_micro  = jiwer.cer(golds, asrs)
wer_norm_micro = jiwer.wer(golds_norm, asrs_norm)
cer_norm_micro = jiwer.cer(golds_norm, asrs_norm)

total_cost = sum(((p.get("raw_response") or {}).get("usage") or {}).get("cost", 0) or 0
                 for p in preds)
total_latency = sum(p.get("latency_s") or 0 for p in preds)

print(f"n scored      : {len(scored)} / {len(preds)}")
print(f"WER  raw      : {wer_raw_micro:.3f}")
print(f"WER  norm     : {wer_norm_micro:.3f}   <-- judge against 0.25")
print(f"CER  raw      : {cer_raw_micro:.3f}")
print(f"CER  norm     : {cer_norm_micro:.3f}")
print(f"total OR cost : ${total_cost:.4f}")
print(f"total latency : {total_latency:.1f} s   avg {total_latency/max(len(preds),1):.2f} s/row")

## Histograms — duration, latency, cost, per-row WER

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
durations = [p.get("duration_s") or 0 for p in preds]
latencies = [p.get("latency_s") or 0 for p in preds]
costs = [((p.get("raw_response") or {}).get("usage") or {}).get("cost", 0) or 0 for p in preds]
wers_norm = [p["wer_norm"] for p in scored if p["wer_norm"] == p["wer_norm"]]  # drop NaN

axes[0, 0].hist(durations, bins=30); axes[0, 0].set_title("duration (s)")
axes[0, 1].hist(latencies, bins=30); axes[0, 1].set_title("OR latency (s)")
axes[1, 0].hist(costs, bins=30); axes[1, 0].set_title("OR cost ($)")
axes[1, 1].hist(wers_norm, bins=30); axes[1, 1].set_title("WER norm per row")
axes[1, 1].axvline(0.25, color="red", linestyle="--", label="green-light 0.25")
axes[1, 1].legend()
plt.tight_layout()
plt.show()

## Triage render — best / worst / random

Each row: **audio widget → gold → ASR → metrics**, stacked vertically for
easy ear-plus-eye comparison. Click play, read the two lines, judge where
Chirp 3 goes wrong.

In [ ]:
def render_row(idx: int, p: dict, bucket: str) -> None:
    audio_fs = ROOT / p["audio_path"]
    header = (
        f"**[{bucket} #{idx+1}]** `{p['instance_id']}` · "
        f"dur={p['duration_s']}s · "
        f"WER={p['wer_norm']:.3f} (raw {p['wer_raw']:.3f}) · "
        f"CER={p['cer_norm']:.3f} (raw {p['cer_raw']:.3f}) · "
        f"lat={p.get('latency_s','?')}s"
    )
    display(Markdown("---"))
    display(Markdown(header))
    if audio_fs.exists():
        display(Audio(filename=str(audio_fs)))
    else:
        display(Markdown(f"_audio missing on disk: `{audio_fs}`_"))
    display(Markdown(f"**gold :** {p.get('text_gold') or '_∅_'}"))
    display(Markdown(f"**asr  :** {p.get('transcript') or '_∅_'}"))


def render_bucket(rows: list[dict], bucket: str) -> None:
    display(Markdown(f"## {bucket} — {len(rows)} rows"))
    for i, p in enumerate(rows):
        render_row(i, p, bucket)


# order the scored predictions by normalised WER; NaN → +inf so they sink
scored_sorted = sorted(scored, key=lambda p: (float("inf") if p["wer_norm"] != p["wer_norm"]
                                              else p["wer_norm"]))
best_rows   = scored_sorted[:N_SHOW]
worst_rows  = [p for p in reversed(scored_sorted) if p["wer_norm"] == p["wer_norm"]][:N_SHOW]
rng = random.Random(RANDOM_SEED)
random_rows = rng.sample(scored, min(N_SHOW, len(scored)))

### Best N (lowest WER_norm)

In [ ]:
render_bucket(best_rows, "BEST")

### Worst N (highest WER_norm)

In [ ]:
render_bucket(worst_rows, "WORST")

### Random N (seed-fixed)

In [ ]:
render_bucket(random_rows, "RANDOM")

## Green-light decision box

- WER_norm ≤ 0.25 → proceed to full ROG (~13.6 k instances) on remote.
- WER_norm > 0.25 → fall-back plan from the handoff: turn-level ASR
  (bundle consecutive same-speaker utterances, re-split via word
  timestamps) or file-level ASR with diarization.